In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
RAW_DIR = "../data/wisdm-dataset/raw"
SUBJECTS = range(1600, 1651)  # 51 subjects

SENSORS = [("phone", "accel"), ("phone", "gyro"), ("watch", "accel"), ("watch", "gyro")]

ACTIVITIES = {
    "A": "Walking",
    "B": "Jogging",
    "C": "Stairs",
    "D": "Sitting",
    "E": "Standing",
    "F": "Typing",
    "G": "Brushing Teeth",
    "H": "Eating Soup",
    "I": "Eating Chips",
    "J": "Eating Pasta",
    "K": "Drinking from Cup",
    "L": "Eating Sandwich",
    "M": "Kicking (Soccer Ball)",
    "O": "Playing Catch w/Tennis Ball",
    "P": "Dribbling (Basketball)",
    "Q": "Writing",
    "R": "Clapping",
    "S": "Folding Clothes",
}

In [ ]:
def get_sensor_data(filepath):
    df = pd.read_csv(
        filepath,
        header=None,
        names=["subject_id", "activity", "timeStamp", "x", "y", "z"],
    )

    df.iloc[:, -1] = df.iloc[:, -1].str.rstrip(";")

    df = df.astype({"z": float})

    df["timeStamp"] = pd.to_datetime(df["timeStamp"], unit="ns")
    df.drop(columns="subject_id", inplace=True)

    return df

In [ ]:
def get_subject_data(subject_id, log):

    dfs: list[pd.DataFrame] = []

    for device, sensor in SENSORS:

        df = get_sensor_data(
            f"{RAW_DIR}/{device}/{sensor}/data_{subject_id}_{sensor}_{device}.txt"
        )

        df["device"] = device

        df = df.rename(
            columns={
                "x": f"x_{sensor}",
                "y": f"y_{sensor}",
                "z": f"z_{sensor}",
            }
        )

        dfs.append(df)

    for df in dfs:
        df.sort_values(by="timeStamp", inplace=True)

    merged_phone = pd.merge_asof(
        dfs[0] if dfs[0].shape[0] < dfs[1].shape[0] else dfs[1],
        dfs[0] if dfs[0].shape[0] >= dfs[1].shape[0] else dfs[1],
        on="timeStamp",
        by=["activity", "device"],
        direction="nearest",
        tolerance=pd.Timedelta("100ms"),
    )

    if log:
        print(f"{merged_phone.shape[0]} rows of phone data")

    # return merged_phone

    merged_watch = pd.merge_asof(
        dfs[2] if dfs[2].shape[0] < dfs[3].shape[0] else dfs[3],
        dfs[2] if dfs[2].shape[0] >= dfs[3].shape[0] else dfs[3],
        on="timeStamp",
        by=["activity", "device"],
        direction="nearest",
        tolerance=pd.Timedelta("100ms"),
    )

    if log:
        print(f"{merged_watch.shape[0]} rows of phone data")

    merged = pd.concat([merged_phone, merged_watch])

    merged = merged.iloc[:, [5, 0, 1, 2, 3, 4, 6, 7, 8]]

    merged["activity"] = merged["activity"].astype("category")
    merged["device"] = merged["device"].astype("category")

    return merged.sort_values(by=["device", "activity", "timeStamp"])

In [ ]:
df = get_subject_data(1600, True)

df.info()

In [ ]:
def load_all_data():

    merged = pd.DataFrame()

    for subject_id in SUBJECTS:

        print(f"loading {subject_id} data...")

        if subject_id == SUBJECTS.start:
            merged = get_subject_data(subject_id, False)
            merged["subject_id"] = np.ones(merged.shape[0], dtype=int) * subject_id
            continue

        other = get_subject_data(subject_id, False)
        other["subject_id"] = np.ones(other.shape[0], dtype=int) * subject_id
        merged = pd.concat([merged, other])

    merged["subject_id"] = merged["subject_id"].astype("category")
    merged["activity"] = merged["activity"].astype("category")

    merged = merged.reindex(
        columns=[
            "subject_id",
            "device",
            "activity",
            "timeStamp",
            "x_accel",
            "y_accel",
            "z_accel",
            "x_gyro",
            "y_gyro",
            "z_gyro",
        ]
    )

    return merged

In [ ]:
df = load_all_data()

In [ ]:
df = df.dropna()

In [ ]:
res = df[(df["activity"] == "E") & (df["device"] == "watch")].sort_values(by="timeStamp").drop(columns=["device", "activity"])

In [ ]:
res.head(80).to_csv("../data/test/test_watch.csv", index=False)

In [ ]:
df[(df["activity"] == "A") & (df["subject_id"] == 1600)].__len__() / df.__len__() * 100

In [ ]:
activity_percent = {}

for activity in ACTIVITIES.keys():
    activity_percent[activity] = (
        df[df["activity"] == activity].__len__() / df.__len__() * 100
    )

activity_percent = pd.DataFrame([activity_percent])

sns.barplot(activity_percent)

In [ ]:
df

In [ ]:
activity_counts = df["activity"].value_counts().sort_index()
plt.figure(figsize=(12, 5))
sns.barplot(
    x=activity_counts.index,
    y=activity_counts.values,
    palette="viridis",
)
plt.title("Number of Samples per Activity")
plt.xlabel("Activity Code")
plt.ylabel("Count")
plt.show()

In [ ]:
test = pd.read_csv("./test_data.csv")
test["timeStamp"] = pd.to_datetime(test["timeStamp"], unit="ns")


In [ ]:
sample = test
sample = sample.sort_values("timeStamp")

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
axes[0].plot(sample["timeStamp"], sample["x_accel"], label="x_accel")
axes[0].plot(sample["timeStamp"], sample["y_accel"], label="y_accel")
axes[0].plot(sample["timeStamp"], sample["z_accel"], label="z_accel")
axes[0].set_title(f"Accelerometer - clapping")
axes[0].legend()

axes[1].plot(sample["timeStamp"], sample["x_gyro"], label="x_gyro")
axes[1].plot(sample["timeStamp"], sample["y_gyro"], label="y_gyro")
axes[1].plot(sample["timeStamp"], sample["z_gyro"], label="z_gyro")
axes[1].set_title(f"Gyroscope - clapping")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Example: subject 1600, activity 'A' (Walking), phone

subj, act = 1604, "D"

sample = df[
    (df["subject_id"] == subj) & (df["activity"] == act) & (df["device"] == "watch")
]
sample = sample.sort_values("timeStamp")

fig, axes = plt.subplots(2, 1, figsize=(15, 8))
axes[0].plot(sample["timeStamp"], sample["x_accel"], label="x_accel")
axes[0].plot(sample["timeStamp"], sample["y_accel"], label="y_accel")
axes[0].plot(sample["timeStamp"], sample["z_accel"], label="z_accel")
axes[0].set_title(f"Accelerometer - {ACTIVITIES[act]}")
axes[0].legend()

axes[1].plot(sample["timeStamp"], sample["x_gyro"], label="x_gyro")
axes[1].plot(sample["timeStamp"], sample["y_gyro"], label="y_gyro")
axes[1].plot(sample["timeStamp"], sample["z_gyro"], label="z_gyro")
axes[1].set_title(f"Gyroscope - {ACTIVITIES[act]}")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Choose one subject and activity
subj, act = 1600, "J"
phone = df[
    (df["subject_id"] == subj) & (df["activity"] == act) & (df["device"] == "phone")
]
watch = df[
    (df["subject_id"] == subj) & (df["activity"] == act) & (df["device"] == "watch")
]

# Align by relative time
phone = phone.sort_values("timeStamp")
watch = watch.sort_values("timeStamp")


phone["rel_time"] = (phone["timeStamp"] - phone["timeStamp"].iloc[0]).dt.total_seconds()
watch["rel_time"] = (watch["timeStamp"] - watch["timeStamp"].iloc[0]).dt.total_seconds()

plt.figure(figsize=(12, 4))
plt.plot(phone["rel_time"], phone["x_accel"], label="Phone x_accel")
plt.plot(watch["rel_time"], watch["x_accel"], label="Watch x_accel", alpha=0.7)
plt.title(f"Phone vs Watch Accelerometer (X-axis) - {ACTIVITIES[act]}")
plt.xlabel("Time (s)")
plt.legend()
plt.show()

In [ ]:
agg_stats = df.groupby("activity")[
    ["x_accel", "y_accel", "z_accel", "x_gyro", "y_gyro", "z_gyro"]
].agg(["mean", "std"])


means = agg_stats.xs("mean", axis=1, level=1)
stds = agg_stats.xs("std", axis=1, level=1)

plt.figure(figsize=(12, 6))
sns.heatmap(
    means[["x_accel", "y_accel", "z_accel"]], annot=True, cmap="coolwarm", fmt=".2f"
)
plt.title("Mean Accelerometer Values per Activity")
plt.show()

In [ ]:
df = df.drop(columns="subject_id")

df

In [ ]:
df.sort_values(by=["device", "activity", "timeStamp"])

In [ ]:
def extract_windows(df, window_sec=2):

    df = df.copy()

    df = df.sort_values(["device", "activity", "timeStamp"])
    df = df.set_index("timeStamp")

    sensor_cols = ["x_accel", "y_accel", "z_accel", "x_gyro", "y_gyro", "z_gyro"]

    fs = 20.0

    def _extract_features(group):

        device, activity, window_start = group.name
        features = {
            "device": device,
            "activity": activity,
            "window_start": window_start,
        }

        for axis in sensor_cols:
            series = group[axis].values
            n = len(series)
            if n == 0:
                continue

            # Time domain features
            mean_val = np.mean(series)
            std_val = std_val = np.std(series, ddof=1) if n > 1 else np.nan
            p25 = np.percentile(series, 25)
            p75 = np.percentile(series, 75)
            kurt = pd.Series(series).kurtosis()
            autocorr = np.corrcoef(series[:-1], series[1:])[0, 1] if n > 1 else np.nan
            rms = np.sqrt(np.mean(np.square(series)))

            # Frequency domain features
            if n > 1:
                fft_vals = np.fft.rfft(series)
                freqs = np.fft.rfftfreq(n, d=1 / fs)
                magnitudes = np.abs(fft_vals)

                if len(magnitudes) > 1:
                    idx_max = np.argmax(magnitudes[1:]) + 1
                    dom_freq = freqs[idx_max]
                    spectral_energy = np.sum(magnitudes[1:] ** 2)
                else:
                    dom_freq = 0.0
                    spectral_energy = 0.0
            else:
                dom_freq = 0.0
                spectral_energy = 0.0

            prefix = axis + "_"
            features[prefix + "mean"] = mean_val
            features[prefix + "std"] = std_val
            features[prefix + "p25"] = p25
            features[prefix + "p75"] = p75
            features[prefix + "kurt"] = kurt
            features[prefix + "autocorr"] = autocorr
            features[prefix + "rms"] = rms
            features[prefix + "dom_freq"] = dom_freq
            features[prefix + "spectral_energy"] = spectral_energy

        return pd.Series(features)

    grouped = df.groupby(["device", "activity", pd.Grouper(freq=f"{window_sec}s")])
    windows = grouped.apply(_extract_features).reset_index(drop=True)
    windows["window_start"] = pd.to_datetime(windows["window_start"])
    return windows

In [ ]:
final_df = extract_windows(df)

In [ ]:
final_df

In [ ]:
final_df = final_df.dropna()
final_df = final_df.drop(columns="window_start")

In [ ]:
df_phone = final_df[final_df["device"] == "phone"]
df_watch = final_df[final_df["device"] == "watch"]

df_phone = df_phone.drop(columns="device")
df_watch = df_watch.drop(columns="device")

In [ ]:
df_phone

In [ ]:
df_phone.skew(numeric_only=True)

In [ ]:
def align_subject_sensors(subject_id):
    """
    Aligns all four sensors for one subject using per‑activity relative time.
    Returns a DataFrame with columns:
        activity, rel_time (timedelta),
        x_phone_accel, y_phone_accel, z_phone_accel,
        x_phone_gyro,  y_phone_gyro,  z_phone_gyro,
        x_watch_accel, y_watch_accel, z_watch_accel,
        x_watch_gyro,  y_watch_gyro,  z_watch_gyro.
    """

    TOLERANCE = pd.Timedelta("100ms")

    sensor_dfs = {}
    for device, sensor in SENSORS:
        filepath = (
            f"{RAW_DIR}/{device}/{sensor}/data_{subject_id}_{sensor}_{device}.txt"
        )
        df = get_sensor_data(filepath)

        df["rel_time"] = df.groupby("activity")["timeStamp"].transform(
            lambda x: (x - x.min())
        )

        suffix = f"{device}_{sensor}"
        df.rename(
            columns={
                "x": f"x_{suffix}",
                "y": f"y_{suffix}",
                "z": f"z_{suffix}",
            },
            inplace=True,
        )

        cols = [
            "activity",
            "rel_time",
            f"x_{suffix}",
            f"y_{suffix}",
            f"z_{suffix}",
        ]
        sensor_dfs[(device, sensor)] = df[cols]

    phone_accel = sensor_dfs[("phone", "accel")]
    phone_gyro = sensor_dfs[("phone", "gyro")]

    phone_merged_list = []

    for act, accel_grp in phone_accel.groupby("activity"):
        if act not in phone_gyro["activity"].values:
            continue

        gyro_grp = phone_gyro[phone_gyro["activity"] == act]

        # Sort by rel_time so merge_asof sees monotonic keys
        accel_grp = accel_grp.sort_values("rel_time")
        gyro_grp = gyro_grp.sort_values("rel_time")

        merged = pd.merge_asof(
            accel_grp if accel_grp.shape[0] < gyro_grp.shape[0] else gyro_grp,
            accel_grp if accel_grp.shape[0] >= gyro_grp.shape[0] else gyro_grp,
            on="rel_time",
            direction="nearest",
            tolerance=TOLERANCE,
        )

        merged.drop(columns=["activity_y"], inplace=True, errors="ignore")

        merged.rename(columns={"activity_x": "activity"}, inplace=True)

        phone_merged_list.append(merged)

    phone_merged = pd.concat(phone_merged_list, ignore_index=True)

    watch_accel = sensor_dfs[("watch", "accel")]
    watch_gyro = sensor_dfs[("watch", "gyro")]

    watch_merged_list = []
    for act, accel_grp in watch_accel.groupby("activity"):
        if act not in watch_gyro["activity"].values:
            continue
        gyro_grp = watch_gyro[watch_gyro["activity"] == act]

        accel_grp = accel_grp.sort_values("rel_time")
        gyro_grp = gyro_grp.sort_values("rel_time")

        merged = pd.merge_asof(
            accel_grp if accel_grp.shape[0] < gyro_grp.shape[0] else gyro_grp,
            accel_grp if accel_grp.shape[0] >= gyro_grp.shape[0] else gyro_grp,
            on="rel_time",
            direction="nearest",
            tolerance=TOLERANCE,
        )
        merged.drop(columns=["activity_y"], inplace=True, errors="ignore")
        merged.rename(columns={"activity_x": "activity"}, inplace=True)
        watch_merged_list.append(merged)

    watch_merged = pd.concat(watch_merged_list, ignore_index=True)

    final_list = []
    for act, phone_grp in phone_merged.groupby("activity"):
        if act not in watch_merged["activity"].values:
            continue
        watch_grp = watch_merged[watch_merged["activity"] == act]

        phone_grp = phone_grp.sort_values("rel_time")
        watch_grp = watch_grp.sort_values("rel_time")

        merged = pd.merge_asof(
            phone_grp if phone_grp.shape[0] < watch_grp.shape[0] else watch_grp,
            phone_grp if phone_grp.shape[0] >= watch_grp.shape[0] else watch_grp,
            on="rel_time",
            direction="nearest",
            tolerance=TOLERANCE,
        )
        merged.drop(columns=["activity_y"], inplace=True, errors="ignore")
        merged.rename(columns={"activity_x": "activity"}, inplace=True)
        final_list.append(merged)

    final = pd.concat(final_list, ignore_index=True)

    # Final clean‑up as in your version
    final.sort_values(["activity", "rel_time"], inplace=True)
    final.reset_index(drop=True, inplace=True)
    return final

In [ ]:
df = align_subject_sensors(1600)
df.info()

In [ ]:
def load_all_data_aligned():

    merged = pd.DataFrame()

    for subject_id in SUBJECTS:

        print(f"loading {subject_id} data...")

        if subject_id == SUBJECTS.start:
            merged = align_subject_sensors(subject_id)
            merged["subject_id"] = np.ones(merged.shape[0], dtype=int) * subject_id
            continue

        other = align_subject_sensors(subject_id)
        other["subject_id"] = np.ones(other.shape[0], dtype=int) * subject_id
        merged = pd.concat([merged, other])

    merged["subject_id"] = merged["subject_id"].astype("category")
    merged["activity"] = merged["activity"].astype("category")

    return merged

In [ ]:
df = load_all_data_aligned().dropna()

In [ ]:
res = df[(df["activity"] == "K") & (df["subject_id"] == 1600)].sort_values(by="rel_time").drop(columns=["activity"])

In [ ]:
res.head(160).to_csv("../data/test/test_aligned(4s).csv", index=False)

In [ ]:
res.head(160).to_csv("../data/test/test_aligned(4s).csv",index=False)

In [ ]:
def extract_windows_aligned(df, window_sec=2):

    df = df.copy()

    df = df.sort_values(["activity", "rel_time"])
    df = df.set_index("rel_time")

    sensor_cols = [
        "x_phone_accel",
        "y_phone_accel",
        "z_phone_accel",
        "x_phone_gyro",
        "y_phone_gyro",
        "z_phone_gyro",
        "x_watch_accel",
        "y_watch_accel",
        "z_watch_accel",
        "x_watch_gyro",
        "y_watch_gyro",
        "z_watch_gyro",
    ]

    fs = 20.0

    def _extract_features(group):

        _, activity, window_start = group.name

        features = {
            "activity": activity,
            "window_start": window_start,
        }

        for axis in sensor_cols:
            series = group[axis].values
            n = len(series)
            if n == 0:
                continue

            # Time domain features
            mean_val = np.mean(series)
            std_val = std_val = np.std(series, ddof=1) if n > 1 else np.nan
            p25 = np.percentile(series, 25)
            p75 = np.percentile(series, 75)
            kurt = pd.Series(series).kurtosis()
            autocorr = np.corrcoef(series[:-1], series[1:])[0, 1] if n > 1 else np.nan
            rms = np.sqrt(np.mean(np.square(series)))

            # Frequency domain features
            if n > 1:
                fft_vals = np.fft.rfft(series)
                freqs = np.fft.rfftfreq(n, d=1 / fs)
                magnitudes = np.abs(fft_vals)

                if len(magnitudes) > 1:
                    idx_max = np.argmax(magnitudes[1:]) + 1
                    dom_freq = freqs[idx_max]
                    spectral_energy = np.sum(magnitudes[1:] ** 2)
                else:
                    dom_freq = 0.0
                    spectral_energy = 0.0
            else:
                dom_freq = 0.0
                spectral_energy = 0.0

            prefix = axis + "_"
            features[prefix + "mean"] = mean_val
            features[prefix + "std"] = std_val
            features[prefix + "p25"] = p25
            features[prefix + "p75"] = p75
            features[prefix + "kurt"] = kurt
            features[prefix + "autocorr"] = autocorr
            features[prefix + "rms"] = rms
            features[prefix + "dom_freq"] = dom_freq
            features[prefix + "spectral_energy"] = spectral_energy

        return pd.Series(features)

    grouped = df.groupby(
        by=["subject_id", "activity", pd.Grouper(freq=f"{window_sec}s")]
    )
    windows = grouped.apply(_extract_features).reset_index(drop=True)
    return windows

In [ ]:
final_df = extract_windows_aligned(df, window_sec=4)

In [ ]:
final_df

In [ ]:
final_df.to_csv(
    "../data/processed/full_device&sensor_aligned_feature_extracted_data(4s).csv", index=False
)